In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-cutting
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Wire cutting untuk estimasi nilai ekspektasi

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Estimasi penggunaan: 22 detik pada prosesor Heron (CATATAN: Ini hanya perkiraan. Waktu eksekusi kamu mungkin berbeda.)*
## Hasil belajar
Setelah mengikuti tutorial ini, pengguna diharapkan memahami:
- Cara menggunakan [`qiskit-addon-cutting`](https://github.com/Qiskit/qiskit-addon-cutting) untuk mempartisi circuit besar menjadi subcircuit yang lebih kecil, sehingga mengurangi efek noise

## Prasyarat
Kami menyarankan pengguna untuk sudah familiar dengan topik berikut sebelum mengikuti tutorial ini:
- Menggunakan primitive [Sampler](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2), yang digunakan dalam alur kerja ini

## Latar Belakang
Circuit-knitting adalah istilah umum yang mencakup berbagai metode untuk mempartisi sebuah circuit menjadi beberapa subcircuit yang lebih kecil dengan lebih sedikit gate atau Qubit. Setiap subcircuit dapat dieksekusi secara independen dan hasil akhirnya diperoleh melalui proses pasca-pemrosesan klasik atas hasil dari setiap subcircuit. Teknik ini dapat diakses melalui [Circuit cutting Qiskit addon](https://qiskit.github.io/qiskit-addon-cutting/index.html); lihat [dokumentasi](https://qiskit.github.io/qiskit-addon-cutting/explanation/index.html) beserta [materi pengantar lainnya](https://qiskit.github.io/qiskit-addon-cutting/tutorials/index.html) untuk penjelasan rinci tentang teknik ini.

Tutorial ini berfokus pada metode yang disebut **wire cutting**, di mana circuit dipartisi sepanjang wire [\[1\], \[2\]](#references). Perlu dicatat bahwa partisi pada circuit klasik cukup sederhana karena keluaran di titik partisi dapat ditentukan secara deterministik, yaitu 0 atau 1. Namun, keadaan Qubit di titik pemotongan umumnya berupa mixed state. Oleh karena itu, setiap subcircuit perlu diukur beberapa kali dalam basis yang berbeda (biasanya basis yang lengkap secara tomografi, seperti basis Pauli [\[3\], \[4\]](#references)) dan disiapkan di eigenstate-nya yang sesuai. Gambar di bawah (courtesy: [\[7\]](#references)) menunjukkan contoh wire cutting untuk GHZ state empat Qubit menjadi tiga subcircuit. Di sini $M_j$ menunjukkan sekumpulan basis (biasanya Pauli X, Y, dan Z), dan $P_i$ menunjukkan sekumpulan eigenstate (biasanya $|0\rangle$, $|1\rangle$, $|+\rangle$, dan $|+i\rangle$).

![wc-1.png](../docs/images/tutorials/wire-cutting/0ce8857b-7f5f-400e-8536-6a496c724d50.avif)
![wc-2.png](../docs/images/tutorials/wire-cutting/cbce4455-4794-4c81-8630-3e3993e1b29f.avif)

Karena setiap subcircuit memiliki lebih sedikit Qubit dan Gate, mereka diharapkan lebih tahan terhadap noise. Tutorial ini menunjukkan contoh di mana metode ini dapat digunakan untuk secara efektif menekan noise dalam sistem.
## Persyaratan
Sebelum memulai tutorial ini, pastikan kamu sudah menginstal hal-hal berikut:

- Qiskit SDK v2.0 atau lebih baru, dengan dukungan [visualisasi](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 atau lebih baru ( `pip install qiskit-ibm-runtime` )
- Circuit cutting Qiskit addon v0.10.0 atau lebih baru (`pip install qiskit-addon-cutting`)
- Qiskit addon utils 0.3 atau lebih baru (`pip install qiskit-addon-utils`)
- Qiskit Aer (`pip install qiskit-aer` )
## Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.circuit import Parameter, ParameterVector, QuantumCircuit
from qiskit.quantum_info import PauliList, SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit.result import sampled_expectation_value

from qiskit_addon_cutting.instructions import CutWire
from qiskit_addon_cutting import (
    cut_wires,
    expand_observables,
    partition_problem,
    generate_cutting_experiments,
    reconstruct_expectation_values,
)

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2, Batch

## Contoh simulator skala kecil
Tutorial ini mengimplementasikan [pola Qiskit](/guides/intro-to-patterns) untuk mensimulasikan circuit Many-Body Localization (MBL) satu dimensi (1D). Circuit MBL adalah circuit yang efisien secara hardware dan diparameterisasi oleh dua parameter $\theta$ dan $\vec{\phi}$. Ketika $\theta$ diset ke $0$ dan keadaan awal disiapkan di $|0\rangle$ untuk semua Qubit, nilai ekspektasi ideal dari $\langle Z_i \rangle$ adalah $+1$ untuk setiap qubit site $i$ terlepas dari nilai-nilai $\vec{\phi}$. Detail lebih lanjut tentang circuit ini tersedia di [artikel](https://www.nature.com/articles/s41467-025-57623-x) ini.

Perlu dicatat bahwa pada simulator tanpa noise, nilai ekspektasi yang diperoleh dengan dan tanpa circuit cutting akan sama.
### Langkah 1: Petakan input klasik ke masalah kuantum
#### Konstruksi circuit MBL 1D
Pertama, kita menyajikan fungsi untuk membangun circuit MBL 1D.

In [2]:
class MBLChainCircuit(QuantumCircuit):
    def __init__(
        self, num_qubits: int, depth: int, use_cut: bool = False
    ) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainCircuit<{num_qubits}, {depth}>"
        )
        evolution = MBLChainEvolution(num_qubits, depth, use_cut)
        self.compose(evolution, inplace=True)


class MBLChainEvolution(QuantumCircuit):
    def __init__(self, num_qubits: int, depth: int, use_cut) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainEvolution<{num_qubits}, {depth}>"
        )

        theta = Parameter("θ")
        phis = ParameterVector("φ", num_qubits)

        for layer in range(depth):
            layer_parity = layer % 2
            # print("layer parity", layer_parity)
            for qubit in range(layer_parity, num_qubits - 1, 2):
                # print(qubit)
                self.cz(qubit, qubit + 1)
                self.u(theta, 0, np.pi, qubit)
                self.u(theta, 0, np.pi, qubit + 1)
                if (
                    use_cut
                    and layer_parity == 0
                    and (
                        qubit == num_qubits // 2 - 1
                        or qubit == num_qubits // 2
                    )
                ):
                    self.append(CutWire(), [num_qubits // 2])
                if use_cut and layer < depth - 1 and layer_parity == 1:
                    if qubit == num_qubits // 2:
                        self.append(CutWire(), [qubit])
            for qubit in range(num_qubits):
                self.p(phis[qubit], qubit)

In [3]:
num_qubits = 10
depth = 2
mbl = MBLChainCircuit(num_qubits, depth)
mbl.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif)

Kita menghitung nilai ekspektasi rata-rata $O = \frac{1}{n} \sum_i Z_i$ atas semua Qubit untuk $\theta = 0$. Karena nilai ekspektasi ideal dari $\langle Z_i \rangle = 1$ $\forall$ $i$, nilai ekspektasi ideal dari $O$ juga $1$. Parameter $\phi$ dipilih secara acak.

In [4]:
np.random.seed(42)
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

Circuit perlu diberi anotasi dengan menyisipkan **CutWire** di lokasi yang diinginkan untuk mempartisinya. Untuk tutorial ini, kita memilih partisi yang sama rata. Circuit MBL dirancang sedemikian rupa sehingga menetapkan `use_cut=True` dalam fungsi akan menyisipkan anotasi dengan benar setelah $\frac{n}{2}$ Qubit, dengan $n$ adalah jumlah Qubit dalam circuit asli. Kita juga menetapkan parameter yang dihasilkan secara acak ke circuit.

In [5]:
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_cut.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/5208e0a8-0.avif)

### Langkah 2: Optimalkan masalah untuk eksekusi hardware kuantum
#### Potong circuit menjadi subcircuit yang lebih kecil
Sekarang kita mempartisi circuit menjadi dua subcircuit yang lebih kecil menggunakan [`qiskit-addon-cutting`](https://qiskit.github.io/qiskit-addon-cutting/). `qiskit-addon-cutting` menambahkan gate virtual `Move` untuk membagi lokasi wire cut dengan menyesuaikan jumlah Qubit secara tepat. Sekarang kita membuat circuit dengan gate virtual ini. Karena ada satu wire cut, jumlah Qubit terkait akan bertambah 1.

In [6]:
mbl_move = cut_wires(mbl_cut)
mbl_move.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1834cb22-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/1834cb22-0.avif)

#### Konstruksi dan perluas observable
Observable, sebagaimana didefinisikan sebelumnya, akan menjadi rata-rata $Z$ pada setiap Qubit. Namun, setelah menyisipkan gate virtual `Move`, jumlah Qubit efektif dalam circuit bertambah. Observable juga harus diperluas untuk memperhitungkan perubahan jumlah Qubit ini. Perlu dicatat bahwa observable selalu bekerja secara trivial (sebagai $I$) pada Qubit ekstra yang ditambahkan untuk gate virtual `Move`.

In [7]:
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
observable

PauliList(['ZIIIIIIIII', 'IZIIIIIIII', 'IIZIIIIIII', 'IIIZIIIIII',
           'IIIIZIIIII', 'IIIIIZIIII', 'IIIIIIZIII', 'IIIIIIIZII',
           'IIIIIIIIZI', 'IIIIIIIIIZ'])

In [8]:
new_obs = expand_observables(observable, mbl, mbl_move)
new_obs

PauliList(['ZIIIIIIIIII', 'IZIIIIIIIII', 'IIZIIIIIIII', 'IIIZIIIIIII',
           'IIIIZIIIIII', 'IIIIIIZIIII', 'IIIIIIIZIII', 'IIIIIIIIZII',
           'IIIIIIIIIZI', 'IIIIIIIIIIZ'])

Now the circuit can be partitioned along the `Move` gate and we obtain the subcircuits, as well as the subobservable, which is the portion of the original observable associated with each subcircuit.

In [12]:
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

Here we visualize the two subcircuits:

In [13]:
subcircuits[0].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1b0e779f-0.avif" alt="Output of the previous code cell" />

In [14]:
subcircuits[1].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/3c802f28-0.avif" alt="Output of the previous code cell" />

Berikut kita visualisasikan dua subcircuit:

In [15]:
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

As discussed before, for each cut the upstream circuit must be measured in a Pauli basis, and the downstream circuit must be prepared in the eigenstate of the basis. The function `generate_cutting_experiments` creates all of these necessary circuits and the coefficients associated with each circuit required for reconstruction. Find more detail in [this paper](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.125.150504).

In [16]:
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/3c802f28-0.avif)

Memperluas observable menggunakan operasi `Move` memerlukan struktur data `PauliList`. Untuk merekonstruksi nilai ekspektasi circuit asli, kita membutuhkan observable dalam format `SparsePauliOp`.

In [17]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)

print(backend)

<IBMBackend('ibm_fez')>


Seperti yang dibahas sebelumnya, untuk setiap potongan circuit upstream harus diukur dalam basis Pauli, dan circuit downstream harus disiapkan dalam eigenstate dari basis tersebut. Fungsi `generate_cutting_experiments` membuat semua circuit yang diperlukan beserta koefisien yang terkait dengan setiap circuit untuk rekonstruksi. Temukan detail lebih lanjut di [paper ini](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.125.150504).

In [20]:
pm_basis = generate_preset_pass_manager(
    optimization_level=2, basis_gates=backend.configuration().basis_gates
)
basis_subexperiments = {
    label: pm_basis.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

In [21]:
sampler = SamplerV2(mode=AerSimulator())
jobs = {
    label: sampler.run(subsystem_subexpts, shots=2**12)
    for label, subsystem_subexpts in basis_subexperiments.items()
}

### Step 4: Post-process and return result in desired classical format

Now we retrieve the result of each subexperiment run and reconstruct the expectation value of the uncut circuit:

In [22]:
# Retrieve results
results = {label: job.result() for label, job in jobs.items()}

In [23]:
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real
reconstructed_expval

np.float64(0.9953821063041687)

In [32]:
methods = [
    "Uncut",
    "Wire cut",
]
values = [
    1,
    reconstructed_expval,
]  # since the ideal expectation value in noiseless simulation is +1

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

### Langkah 4: Pasca-proses dan kembalikan hasil dalam format klasik yang diinginkan
Sekarang kita mengambil hasil setiap subexperiment yang dijalankan dan merekonstruksi nilai ekspektasi circuit yang tidak dipotong:

In [ ]:
num_qubits = 60
depth = 2

# construct the circuit
mbl = MBLChainCircuit(num_qubits, depth)

# create parameters
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

# construct the cut circuit
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_move = cut_wires(mbl_cut)

# Define observable and expand to account for the wire cut
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
new_obs = expand_observables(observable, mbl, mbl_move)

# Construct a SparsePauliOp version of the observable for later use in reconstruction
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

# Partition the circuit and get subcircuits and subobservables
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

# Obtain subexperiments and coefficients
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

# Transpile the subexperiments to the backend
pm = generate_preset_pass_manager(optimization_level=2, backend=backend)
isa_subexperiments = {
    label: pm.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

# Execute the subexperiments and retrieve results
with Batch(backend=backend) as batch:
    sampler = SamplerV2(mode=batch)
    sampler.options.environment.job_tags = ["TUT_WC"]
    jobs = {
        label: sampler.run(subsystem_subexpts, shots=2**12)
        for label, subsystem_subexpts in isa_subexperiments.items()
    }
results = {label: job.result() for label, job in jobs.items()}

# Reconstruct the expectation value of the original observable
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real

# Compute the uncut circuit to obtain the noisy expectation value for comparison
sampler = SamplerV2(mode=backend)
sampler.options.environment.job_tags = ["TUT_WC"]

if mbl.num_clbits == 0:
    mbl.measure_all()
isa_mbl = pm.run(mbl)

pub = (isa_mbl, params)
uncut_job = sampler.run([pub])

uncut_counts = uncut_job.result()[0].data.meas.get_counts()
uncut_expval = sampled_expectation_value(uncut_counts, M_z)

# visualize the results
ax = plt.gca()
methods = ["uncut", "cut"]
values = [uncut_expval, reconstructed_expval]

plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.axhline(y=1, color="k", linestyle="--")
plt.text(0.3, 0.95, "Exact result")
plt.show()

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/37834c72-0.avif" alt="Output of the previous code cell" />

In [29]:
uncut_expval

0.9202473958333336

## Next steps

<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:
- [Periodic boundary conditions with circuit cutting](/docs/tutorials/periodic-boundary-conditions-with-circuit-cutting)
- [Circuit cutting for depth reduction](/docs/tutorials/depth-reduction-with-circuit-cutting)
</Admonition>

## References


[1] Peng, T., Harrow, A. W., Ozols, M., & Wu, X. (2020). Simulating large quantum circuits on a small quantum computer. Physical review letters, 125(15), 150504.

[2] Tang, W., Tomesh, T., Suchara, M., Larson, J., & Martonosi, M. (2021, April). Cutqc: using small quantum computers for large quantum circuit evaluations. In Proceedings of the 26th ACM International conference on architectural support for programming languages and operating systems (pp. 473-486).

[3]  Perlin, M. A., Saleem, Z. H., Suchara, M., & Osborn, J. C. (2021). Quantum circuit cutting with maximum-likelihood tomography. npj Quantum Information, 7(1), 64.

[4]  Majumdar, R., & Wood, C. J. (2022). Error mitigated quantum circuit cutting. arXiv preprint arXiv:2211.13431.

[5]  Khare, T., Majumdar, R., Sangle, R., Ray, A., Seshadri, P. V., & Simmhan, Y. (2023). Parallelizing Quantum-Classical Workloads: Profiling the Impact of Splitting Techniques. In 2023 IEEE International Conference on Quantum Computing and Engineering (QCE) (Vol. 1, pp. 990-1000). IEEE.

[6]  Bhoumik, D., Majumdar, R., Saha, A., & Sur-Kolay, S. (2023). Distributed Scheduling of Quantum Circuits with Noise and Time Optimization. arXiv preprint arXiv:2309.06005.

[7]  Majumdar, R. (2024). Efficient Reduction of Resources and Noise in Discrete Quantum Computing Circuits (Doctoral dissertation, Indian Statistical Institute - Kolkata). https://www.proquest.com/openview/b481def90b1cc80e6b58a77c99e8385c/1?pq-origsite=gscholar&cbl=2026366&diss=y